# 1D Pressure Diffusivity - Fluid flow in Porous Media

This notebook explores the **Pressure Diffusivity Equation**, which is the fundamental PDE for single-phase flow in reservoirs.

## 1. Governing Equation

The 1D diffusivity equation (under small pressure gradients) is:
$$\frac{\partial^2 p}{\partial x^2} = \frac{\phi \mu c_t}{k} \frac{\partial p}{\partial t}$$

Where:
- $p$: Pressure
- $\phi$: Porosity
- $\mu$: Viscosity
- $k$: Permeability
- $c_t$: Total compressibility

## 2. Detailed Analytical Trace (Sympy)

We define the diffusivity constant $\eta = \frac{k}{\phi \mu c_t}$ and solve via separation of variables.

In [ ]:
import sympy as sp
from IPython.display import display, Math

sp.init_printing()

x, t, eta = sp.symbols('x t eta', real=True, positive=True)
P = sp.Function('P')(x, t)

# Governing Equation: d^2P/dx^2 = (1/eta) * dP/dt
diff_eq = sp.Eq(P.diff(x, x), (1/eta) * P.diff(t))
display(Math(f"\\text{{Diffusivity Equation: }} {sp.latex(diff_eq)}"))

# Separation of variables: P(x, t) = X(x)Phi(t)
X = sp.Function('X')(x)
Phi = sp.Function('Phi')(t)
lam = sp.symbols('lambda', real=True, positive=True)

spatial_ode = sp.Eq(X.diff(x, x) + lam**2 * X, 0)
temporal_ode = sp.Eq(Phi.diff(t) + eta * lam**2 * Phi, 0)

sol_x = sp.dsolve(spatial_ode, X)
sol_phi = sp.dsolve(temporal_ode, Phi)

display(Math(f"X(x) = {sp.latex(sol_x.rhs)}"))
display(Math(f"\\Phi(t) = {sp.latex(sol_phi.rhs)}"))

## 3. Numerical Comparison

For $P(0, t) = 500$, $P(L, t) = 100$, and initial pressure $P(x, 0) = 100$. This matches a depletion or injection scenario.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def solve_pressure_numpy(nx, L, nt, dt, eta, u_init):
    dx = L / (nx - 1)
    u = u_init.copy()
    r = eta * dt / dx**2
    for _ in range(nt):
        un = u.copy()
        u[1:-1] = un[1:-1] + r * (un[2:] - 2*un[1:-1] + un[0:-2])
        # Dirichlet BCs for this demo
        u[0] = 500 
        u[-1] = 100
    return u

nx = 101
L = 1000.0 # meters
eta_val = 50.0 # m^2/day (example units)
dt = 1.0
nt = 100
x_vals = np.linspace(0, L, nx)

u0 = 100 * np.ones(nx)
u_final = solve_pressure_numpy(nx, L, nt, dt, eta_val, u0)

plt.figure(figsize=(10, 6))
plt.plot(x_vals, u0, 'k:', label='Initial condition (100)')
plt.plot(x_vals, u_final, 'b-', label='Predicted Pressure Field')
plt.title(f"1D Pressure Profile after {nt} days")
plt.xlabel("Distance (m)")
plt.ylabel("Pressure (psi)")
plt.legend()
plt.grid(True)
plt.show()